# SIR Model with a Dynamic Force of Infection

This notebook has been converted from R to Python for the infectious disease modelling training. It uses **NumPy**, **SciPy**, and **Matplotlib**.

In the previous activity, you gained first experience with the concept of the force of infection. However, the previous model made the unrealistic assumption that the force of infection is constant. In reality, the force of infection, $\lambda$, changes over time depending on the number of infectious people in the population.

For this simple SIR model, we define the force of infection as

$$\\lambda(t) = \\beta\\frac{I(t)}{N}, \\qquad N=S(t)+I(t)+R(t).$$

The differential equations are

$$\\frac{dS}{dt}=-\\lambda S, \\qquad
\\frac{dI}{dt}=\\lambda S-\\gamma I, \\qquad
\\frac{dR}{dt}=\\gamma I.$$

Here, $S$ is the number of susceptible people, $I$ the number of infectious people, and $R$ the number of recovered people. The parameter $\\beta$ is the transmission rate and $\\gamma$ is the recovery rate.

### Question
Write out the differential equations for an SIR model with a dynamic force of infection on paper. Which assumptions are you making in this model structure?

Now simulate the model. Use initial conditions $S=999999$, $I=1$, and $R=0$, a simulation period of 60 days, a recovery rate $\\gamma=0.1$ days$^{-1}$, and a transmission rate $\\beta=1$.

Plot the curves for susceptibility, infection, and recovery over time.

### Question
After how many days does the epidemic peak? What is the peak prevalence?

In [ ]:
# EXERCISE: Complete the model function below.
# Hint: calculate the total population N = S + I + R, then calculate
# the dynamic force of infection lambda_ = beta * I / N.
#
# The function should return the derivatives in this order: dS/dt, dI/dt, dR/dt.

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def sir_model(t, y, beta, gamma):
    S, I, R = y

    # TODO 1: Calculate the total population size
    N = ...

    # TODO 2: Calculate the dynamic force of infection
    lambda_ = ...

    # TODO 3: Define the differential equations
    dS = ...
    dI = ...
    dR = ...

    return [dS, dI, dR]

# Once you have completed sir_model, you can use solve_ivp to solve it.


## Explore different transmission and recovery rates

In the second part of the activity, explore how the epidemic curve changes under different assumptions for $\\beta$ and $\\gamma$. First run the example with the same parameter values as above and check that your output is correct.

Then change the values of $\\beta$ and $\\gamma$ to explore different scenarios.

### Questions
- How does the epidemic pattern change with different values of $\\beta$ or $\\gamma$?
- How do these parameter combinations affect when the epidemic peak occurs, the number of people infectious at the peak, and how long the epidemic lasts?

At this stage, focus on describing what you observe. In a later session, you will explore why these patterns occur.

In [ ]:
# SOLUTION: SIR model with a dynamic force of infection.
# Run this cell after attempting the exercise above. This solution cell
# defines the completed function and the helper used for simulations.

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def sir_model(t, y, beta, gamma):
    S, I, R = y
    N = S + I + R
    lambda_ = beta * I / N

    dS = -lambda_ * S
    dI = lambda_ * S - gamma * I
    dR = gamma * I

    return [dS, dI, dR]


def run_sir_model(beta, gamma, S0=999999, I0=1, R0=0, duration=60, ax=None):
    """Simulate and plot an SIR model for the chosen parameters."""
    initial_values = [S0, I0, R0]
    times = np.arange(0, duration + 1, 1)

    solution = solve_ivp(
        fun=lambda t, y: sir_model(t, y, beta, gamma),
        t_span=(0, duration),
        y0=initial_values,
        t_eval=times,
        rtol=1e-8,
        atol=1e-8
    )

    if not solution.success:
        raise RuntimeError(solution.message)

    S, I, R = solution.y
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(solution.t, S, label="Susceptible (S)", color="black")
    ax.plot(solution.t, I, label="Infectious (I)", color="red")
    ax.plot(solution.t, R, label="Recovered (R)", color="blue")
    ax.set(xlabel="Time (days)", ylabel="Number of people",
           title=f"β = {beta}, γ = {gamma}", ylim=(0, S0 + I0 + R0))
    ax.legend()
    ax.grid(alpha=0.25)

    peak_index = int(np.argmax(I))
    peak_day = solution.t[peak_index]
    peak_infectious = I[peak_index]

    return {
        "time": solution.t,
        "S": S,
        "I": I,
        "R": R,
        "peak_day": peak_day,
        "peak_infectious": peak_infectious
    }

# Example: the same parameter values as in the exercise.
result = run_sir_model(beta=1, gamma=0.1, duration=60)
plt.tight_layout()
plt.show()

print(f"Epidemic peak: day {result['peak_day']:.0f}")
print(f"Peak number infectious: {result['peak_infectious']:.0f}")


In [ ]:
# Compare different combinations of beta and gamma.
# Edit or add combinations below. The 2 x 4 layout provides up to eight plots.

scenarios = [
    (1.0, 0.1, 60),
    (0.5, 0.1, 60),
    (0.3, 0.1, 60),
    (1.0, 0.2, 60),
    (0.5, 0.2, 60),
    (0.3, 0.2, 60),
    (0.8, 0.1, 90),
    (0.2, 0.1, 90),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, (beta, gamma, duration) in zip(axes.flat, scenarios):
    run_sir_model(beta=beta, gamma=gamma, duration=duration, ax=ax)

plt.tight_layout()
plt.show()


## Check your implementation

Use the solution code to confirm that your model code is working properly. If your results differ, check the force of infection and the signs of the three differential equations.

**Note:** The initial recovered population is set to `R0=0`, matching the activity instructions (there is no prior immunity at the start).